In [6]:
from IPython.core import page
from openai.resources.containers.files import content
from playwright.async_api import async_playwright
from pygments.lexers import q
!pip install playwright
!playwright install


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [48]:
# [완성] 백엔드에서 로그인 정보 가져오기
# JAVA 백엔드 없이 임시로 테스트

def get_user_credentials():
    return {
        "login_id": "네이버 로그인 ID",
        "login_pw": "네이버 로그인 PW",
        "session_file": "naver_state.json"
    }

creds = get_user_credentials()
creds

{'login_id': '네이버 로그인 ID',
 'login_pw': '네이버 로그인 PW',
 'session_file': 'naver_state.json'}

In [43]:
# [완성] 자동 로그인

import asyncio
from playwright.async_api import async_playwright
import os

LOGIN_URL = "https://nid.naver.com/nidlogin.login"
BLOG_WRITE_URL = "https://blog.naver.com/{BLOG_ID}?Redirect=Write&"

async def auto_login(login_id, login_pw, session_file):

    """
    네이버 자동 로그인 (세션 기반 + 신규 로그인)
    """

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False, slow_mo=120)

        try:
            if os.path.exists(session_file) and os.path.getsize(session_file) == 0:
                print("세션 파일이 비어 있습니다. 삭제 후 신규 로그인 진행")
                os.remove(session_file)

        except (OSError, FileNotFoundError) :
            # 파일이 이미 삭제되었거나 접근 불가한 경우 무시
            pass

            # 저장된 세션 파일이 존재할 경우 유효성 검사
            if os.path.exists(session_file) :
                print("기존 세션 파일 발견 -> 유효성 검사 중")

                context = await browser.new_context(storage_state=session_file)
                page = await context.new_page()
                await page.goto("https://blog.naver.com", timeout=30000)

            # 로그인 상태인지 검사
                if "nidlogin.login" not in page.url:
                    print("세션 유효 -> 로그인 생략")
                    return True

                print("세션 만료 -> 신규 로그인 필요")

            # 2. 세션이 없거나 만료 -> 최초 자동 로그인
            print(" 네이버 로그인 페이지 이동 ")
            context = await browser.new_context()
            page = await context.new_page()
            await page.goto(LOGIN_URL, timeout=30000)

            print(" 로그인 정보 입력 중 ")

            # 로그인 정보 기입
            await page.fill("#id", login_id)
            await page.fill("#pw", login_pw)

            await page.click("button[type=submit]")
            await page.wait_for_url(lambda url: "nidlogin.login" in url, timeout=10000)
            print(" 새 세션 저장 중 ")
            await context.storage_state(path=session_file)

            print(" 자동 로그인 성공 & 세션 저장 완료 ")
            return True

        except Exception as e:
            print(f"자동 로그인 중 오류 발생: {e}")
            return False
        finally:
            await browser.close()


await auto_login(
    login_id=creds["login_id"],
    login_pw=creds["login_pw"],
    session_file=creds["session_file"]
)


 네이버 로그인 페이지 이동 
 로그인 정보 입력 중 
 새 세션 저장 중 
 자동 로그인 성공 & 세션 저장 완료 


True

In [49]:
# [완성] 자동 업로드

from playwright.async_api import async_playwright
import asyncio

BLOG_ID = "네이버 블로그 ID"

async def write_and_publish(BLOG_ID, title, content,
                            session_file="naver_state.json", headless=True):

    """
    네이버 블로그에 글을 작성하고 발행.

    Returns:
        bool: 발행 성공 여부
    """
    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False, slow_mo=0 if headless else 100)

        try:
            context = await browser.new_context(storage_state=session_file)
            page = await context.new_page()
            print("글쓰기 페이지로 이동중…")
            await page.goto(f"https://blog.naver.com/{BLOG_ID}?Redirect=Write&", timeout=30000)
            await asyncio.sleep(1)

            # mainFrame 진입

            print("mainFrame 로딩중…")
            await page.wait_for_selector("iframe[name='mainFrame']")
            frame = page.frame(name="mainFrame")

            # 1) 기존 작성 팝업 닫기

            print("팝업 대기중…")
            try:
                await frame.wait_for_selector("button.se-popup-button-cancel", timeout=2500)
                print("기존 작성 팝업 발견 → 취소 클릭")
                await frame.click("button.se-popup-button-cancel", force=True)
                await asyncio.sleep(1)
            except TimeoutError:
                print("기존 작성 팝업 없음")
            except Exception as e:
                print(f"팝업 처리 중 예외 발생: {e}")
                raise

            # 2) 도움말 패널 닫기

            print("도움말 패널 확인중…")
            try:
                await frame.wait_for_selector("button.se-help-panel-close-button", timeout=2500)
                print("도움말 패널 발견 → 닫기 클릭")
                await frame.click("button.se-help-panel-close-button", force=True)
                await asyncio.sleep(1)
            except TimeoutError:
                print("도움말 패널 없음")
            except Exception as e:
                print(f"도움말 패널 처리 중 예외 발생: {e}")
                raise

            # 3) 제목 입력

            print("제목 placeholder 찾는 중…")
            await frame.wait_for_selector("p.se-text-paragraph span.se-placeholder")

            print("제목 클릭!")
            await frame.click("p.se-text-paragraph span.se-placeholder")

            print("제목 입력중…")
            await frame.type("p.se-text-paragraph", title)

            print("제목 입력 SUCCESS!")

            # 4) 본문 입력 (수정된 부분)

            print("본문 placeholder 찾는 중…")
            await frame.wait_for_selector(
                "div.se-module-text p.se-text-paragraph span.se-placeholder",timeout=5000)

            print("본문 클릭!")
            await frame.click("div.se-module-text p.se-text-paragraph span.se-placeholder")

            print("본문 입력중…")
            await frame.type("div.se-module-text p.se-text-paragraph", content)

            print("본문 입력 SUCCESS!")

            # 5) 발행 버튼 (1단계)

            print("발행 버튼(1단계) 클릭중…")
            await frame.wait_for_selector("button.publish_btn__m9KHH", timeout=5000)
            await frame.click("button.publish_btn__m9KHH")
            await asyncio.sleep(1)

            # 6) 최종 발행 버튼 (2단계)

            print("최종 발행 버튼(2단계) 클릭중…")
            await frame.wait_for_selector("button[data-testid='seOnePublishBtn']", timeout=5000)
            await frame.click("button[data-testid='seOnePublishBtn']")

            # 발행 완료 확인
            try:
                # 발행 후 URL 변경 또는 성공 메시지 대기
                await page.wait_for_url(lambda url: "/PostView.naver" in url, timeout=10000)
                print("게시물 발행 완료")
                return True
            except Exception as e:
                print(f"발행 완료 확인 실패: {e}")
                return False
            finally:
                await browser.sleep(2)

        except Exception as e:
            print(f"글 작성 중 오류 발생:{e}")
            return False
        finally:
            await browser.close()

In [50]:
# [완성] 자동 로그인 + 자동 업로드 실행

from turtle import title

async def run_auto_login_and_upload(login_id, login_pw, session_file, BLOG_ID, title, content, max_retries=3):
    """
    자동 로그인 후 블로그 포스팅을 수행합니다.

    Args:
        max_retries: 업로드 실패 시 최대 재시도 횟수

    Returns:
        bool: 전체 프로세스 성공 여부
    """

    # 자동 로그인 수행
    print("자동 로그인 시작")
    login_ok = await auto_login(login_id, login_pw, session_file)

    if not login_ok:
        print("자동 로그인 실패 -> 업로드 중단")
        return False

    # 로그인 성공 -> 블로그 글쓰기 페이지 이동 후 업로드 실행
    for attempt in range(max_retries):
        print(f"자동 업로드 시작 (시도 {attempt + 1}/{max_retries})")
        upload_ok = await write_and_publish(BLOG_ID, title, content, session_file)

        if not upload_ok:
            print("전체 자동 업로드 프로세스 완료!")
            return True
        else:
            print(f"업로드 실패 (시도 {attempt + 1}/{max_retries})")
            if attempt < max_retries - 1:
                print("재시도 중")
                await asyncio.sleep(5)

    print("최대 재시도 쵯수 초과 - 업로드 실패")
    return False




In [47]:
login_id = "네이버 로그인 ID"
login_pw = "네이버 로그인 PW"
session_file = "naver_state.json"

BLOG_ID = "네이버 블로그 ID" # blog.naver.com/이 부분

title = "자동 업로드 제목 테스트"
content = "이것은 자동 업로드 본문 테스트입니다!"

await run_auto_login_and_upload(login_id, login_pw, session_file, BLOG_ID, title, content)

 자동 로그인 시작 
 네이버 로그인 페이지 이동 
 로그인 정보 입력 중 
 새 세션 저장 중 
 자동 로그인 성공 & 세션 저장 완료 
 자동 업로드 시작 
글쓰기 페이지로 이동중…
mainFrame 로딩중…
 팝업 대기중…
기존 작성 팝업 없음
도움말 패널 확인중…
도움말 패널 발견 → 닫기 클릭
제목 placeholder 찾는 중…
 제목 클릭!
제목 입력중…
제목 입력 SUCCESS!
본문 placeholder 찾는 중…
본문 클릭!
본문 입력중…
본문 입력 SUCCESS!
발행 버튼(1단계) 클릭중…
최종 발행 버튼(2단계) 클릭중…
게시물 발행 완료!
/n 전체 자동 업로드 프로세스 완료! 
